## Cloud-Based Intelligent Resource Allocation System for Campus Facilities (Phase 1)

**Domain:** Education

**Framework:** Ollama (local LLM)

**Model:** Llama 3 (`llama3:latest`)

**Approach:** Sequential Pipeline

**Purpose:** Waterfall SDLC simulation using Ollama Sequential Pipeline.

## Pre-requisite Setup

**1.** Install Ollama
curl -fsSL https://ollama.ai/install.sh | sh   # Mac/Linux

**Windows:** download from https://ollama.com/download

**2.** Pull Llama 3

ollama pull llama3

**3.** Start Ollama server 

ollama serve

## Environment Setup

Install required libraries and initialize the Ollama local LLM settings.

In [11]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "ollama", "requests", "--quiet"
])
print("Libraries installed.")

Libraries installed.


In [12]:
import os, json, time, re
from datetime import datetime
import ollama

OLLAMA_URL   = "http://127.0.0.1:11434"
MODEL        = "llama3:latest"
TEMPERATURE  = 0.3
NUM_PREDICT  = 16384

token_usage  = {}
total_tokens = 0
start_time   = time.time()

print(f"Model: {MODEL}")
print(f"Framework: Ollama | Approach: Sequential Pipeline")

Model: llama3:latest
Framework: Ollama | Approach: Sequential Pipeline


In [13]:
import math
import re

def validate_phase_metrics(agent_name, response, expected_start):
    errors = []

    # 1. No placeholders
    if "[X]" in response or "[Y]" in response or "[Z]" in response:
        errors.append("Contains placeholders like [X], [Y], [Z]")

    # 2. Extract start and finish
    start_match = re.search(r"Earliest Start:\s*Day\s*(\d+)", response, re.IGNORECASE)
    finish_match = re.search(r"Earliest Finish:\s*Day\s*(\d+)", response, re.IGNORECASE)

    if not start_match:
        errors.append("Missing Earliest Start")
    if not finish_match:
        errors.append("Missing Earliest Finish")

    start_day = None
    finish_day = None

    if start_match:
        start_day = int(start_match.group(1))
        if start_day != expected_start:
            errors.append(f"Wrong Earliest Start: expected Day {expected_start}, got Day {start_day}")

    if finish_match:
        finish_day = int(finish_match.group(1))

    # 3. Validate finish day using inclusive counting
    total_effort_match = re.search(
        r"Total Effort WITH Review:\s*(?:.*?=\s*)?([\d.]+)",
        response,
        re.IGNORECASE
    )

    if start_day is not None and finish_day is not None and total_effort_match:
        total_effort = float(total_effort_match.group(1))
        expected_finish = start_day + math.ceil(total_effort) - 1
        if finish_day != expected_finish:
            errors.append(
                f"Wrong Earliest Finish: expected Day {expected_finish}, got Day {finish_day}"
            )

    # 4. Implementation-specific rule
    if agent_name == "Software_Developer":
        task_match = re.search(r"Total Development Tasks:\s*(\d+)", response, re.IGNORECASE)
        if not task_match:
            errors.append("Missing Total Development Tasks")
        else:
            task_count = int(task_match.group(1))
            if not (25 <= task_count <= 45):
                errors.append(f"Total Development Tasks must be 25–45, got {task_count}")

    return errors

## Project Configuration

Project details and realistic system boundaries are defined here and shared with all agents in the pipeline.

In [14]:
DOMAIN = """
PROJECT: Cloud-Based Intelligent Resource Allocation System for Campus Facilities
DOMAIN: Education
DESCRIPTION: Students and faculty book rooms, labs, and equipment online.
Administrators view real-time occupancy dashboards. AI/ML predicts demand and
auto-allocates resources. Integrates with Google Calendar, Microsoft Outlook,
and SSO (Single Sign-On). Sends booking/conflict notifications. Generates
utilization reports for facility management. Supports 5,000+ concurrent users
and maintains 99.9% uptime.
STAKEHOLDERS: Students, Faculty, Facility Admins, IT Department, University Management
FACILITIES: Classrooms, Computer Labs, Research Labs, Conference Rooms, Study Rooms
"""

PROJECT_SCOPE = """
Project scope and implementation boundaries:
- Total SLOC must be between 12,000 and 22,000
- This is a mid-size cloud-based campus facility management system
- Frontend (React/Next.js): 3,000–5,000 SLOC
- Backend API (Node.js/FastAPI): 4,000–7,000 SLOC
- AI/ML Module: 2,000–4,000 SLOC
- Database layer: 1,500–2,500 SLOC
- Integration layer (SSO, Calendar): 1,000–2,000 SLOC

CONSISTENCY CHECK before printing metrics:
  SLOC-based days  = Total SLOC / 50
  Hours-based days = Sum of all task hours / 8
  These two numbers must be within 15% of each other.
  If not, fix task hours first. Then print:
  CONSISTENCY CHECK: [SLOC days] = [Hours days] ✓
"""

print("Project configuration loaded.")

Project configuration loaded.


## Define Sequential Pipeline Agent

Each agent in the pipeline receives the full output of all previous agents as context. Agents run strictly one after another — no round robin, no orchestrator routing.

In [15]:
def run_agent(name, system_prompt, task, history="", max_retries=3):
    """Invoke one Ollama agent with automatic retry if metrics are missing."""
    global total_tokens

    for attempt in range(1, max_retries + 1):
        if attempt > 1:
            print(f"  [RETRY {attempt}/{max_retries}] {name} — metrics missing, retrying...")

        messages = []
        if history:
            messages.append({"role": "assistant", "content": history[-3000:]})

        # On retry, add extra pressure to include the metrics block
        retry_note = ""
        if attempt > 1:
            retry_note = (
                "\n\nCRITICAL: Your previous response was REJECTED because it was "
                "missing the metrics block at the end. You MUST end your response "
                "with the EXACT metrics block including Earliest Start and "
                "Earliest Finish with real day numbers. No placeholders."
            )

        messages.append({
            "role": "user",
            "content": f"{system_prompt}\n\n{task}{retry_note}"
        })

        response = ollama.chat(
            model=MODEL,
            messages=messages,
            options={
                "temperature": TEMPERATURE,
                "num_predict": NUM_PREDICT,
            }
        )

        output = response["message"]["content"]

        # Check if metrics block exists before accepting
        has_start  = bool(re.search(r"Earliest Start:\s*Day\s*\d+", output, re.IGNORECASE))
        has_finish = bool(re.search(r"Earliest Finish:\s*Day\s*\d+", output, re.IGNORECASE))
        has_effort = bool(re.search(r"Total Effort WITH Review", output, re.IGNORECASE))

        # Customer Proxy and Project Manager don't need metrics
        needs_metrics = name not in ("Customer_Proxy", "Project_Manager")

        if needs_metrics and not (has_start and has_finish and has_effort):
            if attempt < max_retries:
                print(f"  [RETRY] {name} missing metrics block — trying again...")
                continue
            else:
                print(f"  [WARNING] {name} still missing metrics after {max_retries} attempts")

        # Count tokens
        approx_tokens = (
            len(system_prompt.split()) +
            len(task.split()) +
            len(history[-3000:].split()) +
            len(output.split())
        )
        token_usage[name] = token_usage.get(name, 0) + approx_tokens
        total_tokens += approx_tokens

        print(f"  [{name}] ~{approx_tokens:,} tokens")
        return output

    return output 

## Define All AI Agents

Define all 7 agent system prompts for the Waterfall SDLC simulation. Each agent passes its full output to the next agent in the pipeline.

Customer Proxy — presents the problem statement

Project Manager — acknowledges request, outlines 5 Waterfall phases (< 200 words)

Requirements Engineer — ≥ 15 use cases, ≥ 40 FRs, ≥ 15 NFRs — 5 req/day

System Engineer — architecture, diagrams, ERD, tech stack — 5 pages/day

Software Developer — 25–45 tasks, SLOC 12,000–22,000 — 50 SLOC/day

Test Engineer — functional + non-functional test cases — 2 TC/day

Documentation Engineer — end-user, developer, training docs — 3 pages/day

In [16]:
# ── Agent 1: Customer Proxy ──────────────────────────────────────────────────
customer_proxy_prompt = f"""
You are the Customer Proxy AI Agent for the Waterfall SDLC.
Your role is to clearly restate and structure the customer's requirements
so the Project Manager fully understands the scope of the system.
{DOMAIN}
"""

# ── Agent 2: Project Manager ─────────────────────────────────────────────────
project_manager_prompt = f"""
You are the Project Manager AI Agent for the Waterfall SDLC.
You have received the customer request. Briefly acknowledge it, state the 5 Waterfall
phases that will be executed (Requirements, Design, Implementation, Testing, Documentation),
and confirm you are delegating to each specialist agent in order.
Keep your response concise — under 200 words.
{DOMAIN}
"""

# ── Agent 3: Requirements Engineer ───────────────────────────────────────────
requirement_engineer_prompt = f"""
You are the Requirements Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Create a COMPLETE use case list with unique IDs (UC-001, UC-002, ...) — at least 15 use cases.
   For each use case provide: Use Case ID, Name, Primary Actor, Preconditions,
   Postconditions, Main Flow, Alternate Flows.
2. Create FUNCTIONAL requirements (FR-001, FR-002, ...) — at least 40 requirements.
3. Create NON-FUNCTIONAL requirements (NFR-001, NFR-002, ...) — at least 15 requirements.
4. All requirement IDs are sequential across the entire document.
5. Include review and rework tasks (10% buffer).

STRICT CONSTRAINTS:
- You MUST fill in ALL fields in the metrics block with real numbers.
- Total Use Cases MUST show a number (minimum 15).
- Total Functional Requirements MUST show a number (minimum 40).
- Total Non-Functional Requirements MUST show a number (minimum 15).
- Number of Tasks MUST show a number between 6 and 10.
  Tasks include: Requirements Gathering, Use Case Documentation,
  FR Documentation, NFR Documentation, Requirements Review, Requirements Rework.
- Do NOT leave any field blank or empty.

Then calculate effort EXACTLY as follows:
- Step 1: Work = Total FR + NFR only (do NOT include use cases in the count)
- Step 2: Productivity rate = 5 requirements per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
- Step 6: Earliest Start = Day 1
- Step 7: Earliest Finish = Day 1 + Total Effort WITH Review - 1 (round up to whole number)

End your response with EXACTLY this block (real numbers only, no placeholders):
=== REQUIREMENTS PHASE METRICS ===
Total Use Cases:
Total Functional Requirements (FR):
Total Non-Functional Requirements (NFR):
Total Requirements (FR + NFR):
Productivity Rate: 5 requirements per day
Base Effort: [Total REQs] / 5 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day 1
Earliest Finish: Day [X]
Number of Tasks in Requirements Phase:
================================
"""

# ── Agent 4: System Engineer ──────────────────────────────────────────────────
system_engineer_prompt = f"""
You are the System Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Create a HIGH-LEVEL SYSTEM ARCHITECTURE with layered design:
   (Client Layer, API/Orchestration Layer, AI/Intelligence Layer,
    Storage Layer, Cross-Cutting Services)
2. Provide a COMPONENT DIAGRAM (textual description)
3. Provide a SEQUENCE DIAGRAM for the Room Booking use case
4. Create an ENTITY-RELATIONSHIP DIAGRAM (textual) with all entities
5. Describe the TECHNOLOGY STACK recommendations
6. Design the AI/ML demand prediction module architecture
7. Design SSO and Calendar integration architecture
8. Reference specific FR and NFR IDs from the requirements document
9. Include review and rework tasks (10% buffer)

STRICT CONSTRAINTS:
- Earliest Start = Requirements Earliest Finish + 1
  EXAMPLE: If Requirements Finish = Day 9, then Design Start = Day 10
  NEVER use Day 1 as start. Always add 1 to previous phase finish day.
- Review effort = Base Effort × 10%
  EXAMPLE: Base Effort = 4 days → Review = 4 × 0.10 = 0.4 days
- Earliest Finish = Earliest Start + ceil(Total Effort) - 1
  EXAMPLE: Start = Day 10, Effort = 4.4 → ceil = 5 → Finish = 10+5-1 = Day 14

Then calculate effort EXACTLY:
- Step 1: Work = estimated total number of pages in design document
- Step 2: Productivity rate = 5 pages per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
- Step 6: Earliest Start = Requirements Earliest Finish + 1
- Step 7: Earliest Finish = Earliest Start + Total Effort WITH Review - 1 (round up to whole number)

End with EXACTLY:
=== DESIGN PHASE METRICS ===
Total Design Document Pages:
Architecture Diagrams Count:
Database Schema Pages:
API Documentation Pages:
UI Wireframe Pages:
Other Design Pages:
Productivity Rate: 5 pages per day
Base Effort: [Total Pages] / 5 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Design Phase:
============================
"""

# ── Agent 5: Software Developer ───────────────────────────────────────────────
software_developer_prompt = f"""
You are the Software Developer AI Agent for the Waterfall SDLC.
{DOMAIN}
{PROJECT_SCOPE}

Your responsibilities:
1. List ALL development tasks with IDs (T-001, T-002, ...)
2. For each task: ID, Module, Description, SLOC estimate, Hours estimate, Linked FR IDs
3. Organize tasks by module:
   - User Auth & SSO Module
   - Facility Booking Module
   - Real-time Occupancy Dashboard
   - AI Demand Prediction Module
   - Notification & Alerts Module
   - Reporting & Analytics Module
   - Admin Management Module
   - API Gateway & Integration
4. Minimum 25 tasks, maximum 45 tasks
5. Include code review tasks (10% buffer)

STRICT CONSTRAINTS:
- Generate EXACTLY between 25 and 45 tasks. Do NOT exceed 45.
  Count as you write: T-001, T-002...T-025 minimum. Do NOT stop before T-025.
- Each SLOC component MUST have ONE specific number — not a range.
- Frontend: pick one number between 3,000–5,000
- Backend: pick one number between 4,000–7,000
- Database: pick one number between 1,500–2,500
- AI/ML: pick one number between 2,000–4,000
- Integration: pick one number between 1,000–2,000
- After writing all SLOC values, add them up:
  Frontend + Backend + Database + AI/ML + Integration = Total SLOC
  The Total SLOC field MUST equal this sum exactly.
- Review effort = Base Effort × 10%
  EXAMPLE: Base Effort = 360 days → Review = 360 × 0.10 = 36 days NOT 3.6 days
- Total Effort = Base Effort + Review effort
  EXAMPLE: 360 + 36 = 396 days NOT 39.6 days
- Consistency check = Total SLOC / 50 × 8
  EXAMPLE: 18,000 / 50 × 8 = 2,880 hours NOT 288 hours
  Compare 2,880 to Total Task Hours. If within 15% → PASS else → FAIL
- You MUST verify your SLOC sum before writing Total SLOC:
  Write out: Frontend + Backend + Database + AI/ML + Integration = ?
  EXAMPLE: 3,500 + 6,000 + 2,000 + 3,500 + 1,500 = 16,500
  Total SLOC MUST equal this sum. MUST be between 12,000 and 22,000.
- You MUST write at least 25 tasks before stopping.
  After writing T-020, keep going: T-021, T-022, T-023, T-024, T-025.
  Do NOT submit until you have written T-025 minimum.
- Earliest Finish = Earliest Start + Total Effort WITH Review - 1
  EXAMPLE: Start=Day 21, Total Effort=528 days
  Finish = 21 + 528 - 1 = 548
  NOT 541. Add all three numbers carefully.  

Then calculate effort EXACTLY:
- Step 1: Work = total SLOC (MUST be between 12,000 and 22,000)
- Step 2: Productivity rate = 50 SLOC per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
- Step 6: Sum all task hours once. Then check: SLOC/50 × 8 ≈ Total Task Hours
- Step 7: Earliest Start = Design Earliest Finish + 1
- Step 8: Earliest Finish = Earliest Start + Total Effort WITH Review - 1 (round up to whole number)

End with EXACTLY:
=== IMPLEMENTATION PHASE METRICS ===
Total Development Tasks:
Frontend SLOC:
Backend SLOC:
Database Scripts SLOC:
AI/ML Module SLOC:
Integration/API SLOC:
Total SLOC (MUST be 12,000–22,000):
Productivity Rate: 50 SLOC per day
Base Effort: [Total SLOC] / 50 = [X] days
Code Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Total Task Hours (sum of all tasks):
CONSISTENCY CHECK: [SLOC/50 × 8] = [X] hrs ≈ Total Task Hours [PASS/FAIL]
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Implementation Phase:
===================================
"""

# ── Agent 6: Test Engineer ────────────────────────────────────────────────────
test_engineer_prompt = f"""
You are the Test Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Define the overall TEST STRATEGY (levels: unit, integration, system, AI evaluation)
2. Create FUNCTIONAL TEST CASES (TC-001 through TC-XX) with:
   Related UC/FR IDs, Preconditions, Steps, Expected Results
3. Create NON-FUNCTIONAL TEST CASES (NTC-001 through NTC-XX) covering:
   Performance (5,000 concurrent users), AI Accuracy (>85% target),
   Availability (99.9% uptime), Security/RBAC, Usability
4. Provide a TESTING SCHEDULE broken into weeks
5. Include test review tasks (10% buffer)

STRICT CONSTRAINTS:
- Fill ALL fields with real numbers. No N/A, no blanks, no zeros for UAT.
- UAT MUST be at least 5.
- Follow this EXACT structure:
  Functional TC = Unit + Integration + System/E2E
  Non-Functional NTC = Performance + AI Accuracy + Security/RBAC + Availability + Usability
  Total Test Cases = Functional TC + Non-Functional NTC
  VERIFY: All three sums must be correct before writing metrics block.
  EXAMPLE: Unit(10)+Integration(8)+System(7) = Functional TC(25)
           Perf(5)+AI(5)+Security(5)+Avail(5)+Usability(5) = NTC(25)
           Total = 25+25 = 50

Then calculate effort EXACTLY:
- Step 1: Work = total number of test cases
- Step 2: Productivity rate = 2 test cases per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
- Step 6: Earliest Start = Implementation Earliest Finish + 1
- Step 7: Earliest Finish = Earliest Start + Total Effort WITH Review - 1(round up to whole number)

End with EXACTLY:
=== TESTING PHASE METRICS ===
Total Test Cases:
Functional Test Cases (TC):
Non-Functional Test Cases (NTC):
Unit Test Cases:
Integration Test Cases:
System/E2E Test Cases:
Performance Test Cases:
AI Accuracy Test Cases:
UAT Test Cases:
Productivity Rate: 2 test cases per day
Base Effort: [Total TCs] / 2 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Testing Phase:
============================
"""

# ── Agent 7: Documentation Engineer ──────────────────────────────────────────
documentation_engineer_prompt = f"""
You are the Documentation Engineer AI Agent for the Waterfall SDLC.
{DOMAIN}

Your responsibilities:
1. Create an END-USER DOCUMENTATION OUTLINE:
   - Student User Guide (booking, equipment, notifications)
   - Faculty User Guide (booking, recurring reservations)
   - Administrator Guide (dashboard, reports, AI settings)
   - AI System Explainer (how demand prediction works)
   - Troubleshooting & FAQ
2. Create a DEVELOPER DOCUMENTATION OUTLINE:
   - System Architecture Overview
   - RESTful & GraphQL API Reference
   - Database Schema & ERD
   - AI/ML Model Documentation
   - Deployment & CI/CD Guide
   - Security & Compliance Guide
3. Create TRAINING MATERIALS outline:
   - Student Onboarding (pages estimate)
   - Admin Training Manual (pages estimate)
4. Estimate pages per section
5. Include documentation review tasks (10% buffer)

STRICT CONSTRAINTS:
- Number of Tasks in Documentation Phase MUST show a real number between 8 and 15.
- Do NOT leave the tasks field blank.
- Tasks include: Write Student Guide, Write Faculty Guide, Write Admin Guide,
  Write AI Explainer, Write API Reference, Write Architecture Docs,
  Write AI/ML Docs, Write CI/CD Guide, Write Training Materials, Review Docs.

Then calculate effort EXACTLY:
- Step 1: Work = total documentation pages
- Step 2: Productivity rate = 3 pages per day
- Step 3: Effort = Work / Productivity rate
- Step 4: Review effort = Effort × 10%
- Step 5: Total Effort WITH Review = Effort + Review effort
- Step 6: Earliest Start = Testing Earliest Finish + 1
- Step 7: Earliest Finish = Earliest Start + Total Effort WITH Review - 1 (round up to whole number)

End with EXACTLY:
=== DOCUMENTATION PHASE METRICS ===
End-User Documentation Pages:
Technical/Developer Documentation Pages:
Training Material Pages:
Total Documentation Pages:
Productivity Rate: 3 pages per day
Base Effort: [Total Pages] / 3 = [X] days
Review Effort (10%): [X] days
Total Effort WITH Review: [X] days
Earliest Start: Day [X]
Earliest Finish: Day [X]
Number of Tasks in Documentation Phase:
===================================
"""

print("All 7 agent prompts defined successfully.")
print("  - Customer_Proxy         (Sequential Pipeline)")
print("  - Project_Manager        (Sequential Pipeline)")
print("  - Requirement_Engineer   (Sequential Pipeline)")
print("  - System_Engineer        (Sequential Pipeline)")
print("  - Software_Developer     (Sequential Pipeline)")
print("  - Test_Engineer          (Sequential Pipeline)")
print("  - Documentation_Engineer (Sequential Pipeline)")

All 7 agent prompts defined successfully.
  - Customer_Proxy         (Sequential Pipeline)
  - Project_Manager        (Sequential Pipeline)
  - Requirement_Engineer   (Sequential Pipeline)
  - System_Engineer        (Sequential Pipeline)
  - Software_Developer     (Sequential Pipeline)
  - Test_Engineer          (Sequential Pipeline)
  - Documentation_Engineer (Sequential Pipeline)


## Customer Problem Statement

The customer sends the initial requirements into the pipeline. This becomes the starting input for the Project Manager.

In [17]:
customer_message = (
    "I want to create a Cloud-Based Intelligent Resource Allocation System "
    "for Campus Facilities at our university. The system must:\n"
    "1. Allow students and faculty to book rooms, labs, and equipment online\n"
    "2. Allow administrators to view real-time occupancy dashboards\n"
    "3. Use AI/ML to predict demand and auto-allocate resources\n"
    "4. Integrate with Google Calendar, Microsoft Outlook, and SSO\n"
    "5. Send notifications for booking confirmations and conflicts\n"
    "6. Generate utilization reports for facility management\n"
    "7. Support 5,000+ concurrent users and maintain 99.9% uptime\n"
    "Please generate the complete Waterfall project plan."
)

print("Customer message defined.")
print(f"Message length: {len(customer_message)} characters")

Customer message defined.
Message length: 615 characters


## Run Experiment

Each agent runs sequentially. The full output of every previous agent is appended to history and passed as context to the next agent, forming a strict sequential pipeline.

In [18]:
## Run Experiment

def extract_finish_day(text):
    """Parse 'Earliest Finish: Day X' from an agent's metrics block."""
    patterns = [
        r"Earliest Finish:\s*Day\s*(\d+)",
        r"Earliest Finish.*?(\d+)",
        r"Finish.*?Day\s*(\d+)"
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return int(match.group(1))
    return None


print("Starting Ollama Sequential Pipeline experiment...")
print("=" * 60)

start_time        = time.time()
history           = ""
all_responses     = {}
phase_finish_days = {}

agents_in_order = [
    ("Customer_Proxy",         customer_proxy_prompt),
    ("Project_Manager",        project_manager_prompt),
    ("Requirement_Engineer",   requirement_engineer_prompt),
    ("System_Engineer",        system_engineer_prompt),
    ("Software_Developer",     software_developer_prompt),
    ("Test_Engineer",          test_engineer_prompt),
    ("Documentation_Engineer", documentation_engineer_prompt),
]

for i, (agent_name, system_prompt) in enumerate(agents_in_order):
    print(f"\nStep {i + 1}/7 → {agent_name}")
    print("-" * 50)

    # ── Build task ────────────────────────────────────────────────────────────
    if agent_name == "Customer_Proxy":
        task = customer_message
    else:
        prev_finish    = list(phase_finish_days.values())[-1] if phase_finish_days else 0
        expected_start = prev_finish + 1

        task = (
            f"Continue the Waterfall SDLC process.\n\n"
            f"STRICT RULES:\n"
            f"- Base your work ONLY on the previous phase outputs shown below\n"
            f"- Maintain full consistency with all values from earlier phases\n"
            f"- Do NOT invent new numbers that conflict with prior metrics\n"
            f"- ALL calculations must be arithmetically accurate\n"
            f"- Your Earliest Start MUST be Day {expected_start}\n\n"
            f"Previous Phase Outputs (last 3000 chars):\n"
            f"{history[-3000:]}"
        )

        print(f"  [Expected Start] Day {expected_start}")

    # ── Run agent ─────────────────────────────────────────────────────────────
    response = run_agent(agent_name, system_prompt, task, history)

    # ── Extract finish day ────────────────────────────────────────────────────
    finish_day = extract_finish_day(response)
    if finish_day:
        phase_finish_days[agent_name] = finish_day
        print(f"  [{agent_name}] Finish day extracted: Day {finish_day}")
    else:
        print(f"  [WARNING] Could not extract finish day for {agent_name}")

    # ── Validate metrics ──────────────────────────────────────────────────────
    if agent_name not in ("Customer_Proxy", "Project_Manager"):
        errors = validate_phase_metrics(agent_name, response, expected_start)
        if errors:
            print(f"  ⚠️ Validation warnings for {agent_name}:")
            for err in errors:
                print(f"      - {err}")
        else:
            print(f"  ✅ Validation passed for {agent_name}")

    # ── Accumulate history and store response ─────────────────────────────────
    history                   += f"\n\n{'='*40}\n{agent_name}:\n{response}"
    all_responses[agent_name]  = response
    print(f"  {agent_name} responded ({len(response):,} chars)")

total_elapsed = time.time() - start_time

print("\n" + "=" * 60)
print(f"Sequential Pipeline completed in {total_elapsed:.1f}s")
print("=" * 60)

Starting Ollama Sequential Pipeline experiment...

Step 1/7 → Customer_Proxy
--------------------------------------------------
  [Customer_Proxy] ~696 tokens
  [WARNING] Could not extract finish day for Customer_Proxy
  Customer_Proxy responded (3,596 chars)

Step 2/7 → Project_Manager
--------------------------------------------------
  [Expected Start] Day 1
  [Project_Manager] ~1,221 tokens
  [WARNING] Could not extract finish day for Project_Manager
  Project_Manager responded (1,446 chars)

Step 3/7 → Requirement_Engineer
--------------------------------------------------
  [Expected Start] Day 1
  [RETRY] Requirement_Engineer missing metrics block — trying again...
  [RETRY 2/3] Requirement_Engineer — metrics missing, retrying...
  [Requirement_Engineer] ~1,620 tokens
  [Requirement_Engineer] Finish day extracted: Day 13
  ✅ Validation passed for Requirement_Engineer
  Requirement_Engineer responded (2,042 chars)

Step 4/7 → System_Engineer
--------------------------------------

## Save Results

Save each agent's output as a UTF-8 encoded markdown file and a combined report to Phase1_Experiment3_Results_Ollama/. Also saves an experiment metadata JSON.

In [19]:
output_dir = "Phase1_Experiment3_Results_Ollama"
os.makedirs(output_dir, exist_ok=True)

# ── Save individual agent outputs ─────────────────────────────────────────────
for agent_name, response in all_responses.items():
    filepath = os.path.join(output_dir, f"{agent_name}_output.md")
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"# {agent_name.replace('_', ' ')} Output\n\n")
        f.write(f"**Model:** {MODEL} | **Tokens:** ~{token_usage.get(agent_name, 0):,}\n\n")
        f.write(response)
    print(f"Saved: {filepath}")

# ── Save combined report ──────────────────────────────────────────────────────
section_names = {
    "Customer_Proxy":         "SECTION 0 ║ CUSTOMER PROXY",
    "Project_Manager":        "SECTION 1 ║ PROJECT MANAGER",
    "Requirement_Engineer":   "SECTION 2 ║ REQUIREMENTS ENGINEER",
    "System_Engineer":        "SECTION 3 ║ SYSTEM ENGINEER",
    "Software_Developer":     "SECTION 4 ║ SOFTWARE DEVELOPER",
    "Test_Engineer":          "SECTION 5 ║ TEST ENGINEER",
    "Documentation_Engineer": "SECTION 6 ║ DOCUMENTATION ENGINEER",
}

combined_path = os.path.join(output_dir, "Phase1_Waterfall_Complete_Report.md")
with open(combined_path, "w", encoding="utf-8") as f:
    f.write("# Phase 1 ║ Waterfall Project Plan\n")
    f.write("# Cloud-Based Intelligent Resource Allocation System for Campus Facilities\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"**LLM Model:** {MODEL}\n")
    f.write(f"**Framework:** Ollama — Sequential Pipeline\n\n---\n\n")
    f.write("## SECTION 0 ║ CUSTOMER PROBLEM STATEMENT\n\n")
    f.write(customer_message.strip() + "\n\n---\n\n")
    for agent_name, response in all_responses.items():
        label = section_names.get(agent_name, agent_name)
        f.write(f"## {label}\n\n")
        f.write(f"**Model:** {MODEL} | **Tokens:** ~{token_usage.get(agent_name, 0):,}\n\n")
        f.write(response)
        f.write("\n\n---\n\n")

print(f"\nSaved combined report: {combined_path}")

# ── Save metadata JSON ────────────────────────────────────────────────────────
metadata = {
    "experiment": {
        "title":               "Phase 1 ║ Waterfall Project Planning",
        "project":             "Cloud-Based Intelligent Resource Allocation System",
        "model":               MODEL,
        "framework":           "Ollama",
        "interaction_type":    "Sequential Pipeline",
        "temperature":         TEMPERATURE,
        "date":                datetime.now().isoformat(),
        "total_tokens":        total_tokens,
        "total_time_seconds":  round(total_elapsed, 2),
    },
    "agents": [
        {"name": k, "tokens": token_usage.get(k, 0), "model": MODEL}
        for k in all_responses.keys()
    ],
    "sequence": {
        "type":          "sequential_pipeline",
        "total_agents":  len(all_responses),
        "interaction":   "Each agent reads the last 3,000 characters of the accumulated outputs",
    },
}

metadata_path = os.path.join(output_dir, "experiment_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata: {metadata_path}")

Saved: Phase1_Experiment3_Results_Ollama/Customer_Proxy_output.md
Saved: Phase1_Experiment3_Results_Ollama/Project_Manager_output.md
Saved: Phase1_Experiment3_Results_Ollama/Requirement_Engineer_output.md
Saved: Phase1_Experiment3_Results_Ollama/System_Engineer_output.md
Saved: Phase1_Experiment3_Results_Ollama/Software_Developer_output.md
Saved: Phase1_Experiment3_Results_Ollama/Test_Engineer_output.md
Saved: Phase1_Experiment3_Results_Ollama/Documentation_Engineer_output.md

Saved combined report: Phase1_Experiment3_Results_Ollama/Phase1_Waterfall_Complete_Report.md
Saved metadata: Phase1_Experiment3_Results_Ollama/experiment_metadata.json


## Experiment Summary

Displays the final per-agent token breakdown and extracts the metrics block from each Waterfall phase for a quick comparative view.

In [22]:
print("=" * 60)
print("  EXPERIMENT 3 COMPLETE")
print("=" * 60)
print(f"  Model:        {MODEL}")
print(f"  Framework:    Ollama")
print(f"  Interaction:  Sequential Pipeline")
print(f"  Total Agents: {len(all_responses)}")
print(f"  Total Steps:  7 (one per Waterfall phase)")
print(f"  Total Tokens: ~{total_tokens:,}")
print(f"  Total Time:   {total_elapsed:.1f}s")
print()
print("  Agent Breakdown:")
for agent_name, response in all_responses.items():
    tokens = token_usage.get(agent_name, 0)
    print(f"    {agent_name:30s} | Tokens: ~{tokens:6,} | Response: {len(response):6,} chars")
print()
print(f"  Output Directory: {output_dir}/")
print("=" * 60)

# ── Per-phase metrics extraction ──────────────────────────────────────────────
print()
print("=" * 60)
print("  PHASE METRICS SUMMARY")
print("=" * 60)

metrics_patterns = {
    "Requirement_Engineer":   r"=== REQUIREMENTS PHASE METRICS ===(.*?)==================================",
    "System_Engineer":        r"=== DESIGN PHASE METRICS ===(.*?)==================================",
    "Software_Developer":     r"=== IMPLEMENTATION PHASE METRICS ===(.*?)==================================",
    "Test_Engineer":          r"=== TESTING PHASE METRICS ===(.*?)==================================",
    "Documentation_Engineer": r"=== DOCUMENTATION PHASE METRICS ===(.*?)==================================",
}

for agent_name, pattern in metrics_patterns.items():
    print(f"\n[{agent_name}]")
    print("-" * 50)
    if agent_name in all_responses:
        # Try strict match first
        match = re.search(pattern, all_responses[agent_name], re.DOTALL)
        if match:
            print(match.group(0).strip())
        else:
            # Try flexible match — Llama 3 sometimes omits trailing ===
            flexible_pattern = pattern.replace("==================================", "(?:==+|\\*+|$)")
            match = re.search(flexible_pattern, all_responses[agent_name], re.DOTALL)
            if match:
                print(match.group(0).strip())
            else:
                # Last resort — just find the header and print next 500 chars
                header_map = {
                    "Requirement_Engineer": "REQUIREMENTS PHASE METRICS",
                    "System_Engineer": "DESIGN PHASE METRICS",
                    "Software_Developer": "IMPLEMENTATION PHASE METRICS",
                    "Test_Engineer": "TESTING PHASE METRICS",
                    "Documentation_Engineer": "DOCUMENTATION PHASE METRICS",
                }
                header = header_map[agent_name]
                idx = all_responses[agent_name].find(header)
                if idx != -1:
                    print(all_responses[agent_name][idx-4:idx+600].strip())
                else:
                    print("❌ Metrics block NOT found — showing last 400 chars:")
                    print(all_responses[agent_name][-400:].strip())
    else:
        print("❌ No response found for this agent")

print("=" * 60)

  EXPERIMENT 3 COMPLETE
  Model:        llama3:latest
  Framework:    Ollama
  Interaction:  Sequential Pipeline
  Total Agents: 7
  Total Steps:  7 (one per Waterfall phase)
  Total Tokens: ~10,801
  Total Time:   566.8s

  Agent Breakdown:
    Customer_Proxy                 | Tokens: ~   696 | Response:  3,596 chars
    Project_Manager                | Tokens: ~ 1,221 | Response:  1,446 chars
    Requirement_Engineer           | Tokens: ~ 1,620 | Response:  2,042 chars
    System_Engineer                | Tokens: ~ 1,760 | Response:  3,155 chars
    Software_Developer             | Tokens: ~ 2,061 | Response:  2,695 chars
    Test_Engineer                  | Tokens: ~ 1,763 | Response:  2,159 chars
    Documentation_Engineer         | Tokens: ~ 1,680 | Response:  1,539 chars

  Output Directory: Phase1_Experiment3_Results_Ollama/

  PHASE METRICS SUMMARY

[Requirement_Engineer]
--------------------------------------------------
=== REQUIREMENTS PHASE METRICS ===
Total Use Cases: 15
T